In [54]:
import json

from docx import Document
from docx.oxml.ns import qn
from docx.shared import Pt


class TRFUpdater:
    """Update TRF JSON + DOCX using values from CDR JSON."""

    APPLICANT_FIELDS = {
        "applicant",
        "applicant’s name\t:",
        "applicant's name\t:"
    }

    MANUFACTURER_FIELDS = {
        "manufacturer",
        "manufacturer 1",
        "manufacturer\t:",
        "manufacturer :",
        "manufacturer:"
    }

    @staticmethod
    def load_json(path: str) -> dict:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)

    @staticmethod
    def save_json(data: dict, path: str) -> None:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4, ensure_ascii=False)

    @staticmethod
    def clean(text) -> str:
        return (text or "").strip().lower()

    @classmethod
    def extract_cdr_values(cls, cdr_data: dict) -> tuple:
        """Extract applicant + manufacturer from CDR."""

        applicant = manufacturer = None

        items = cdr_data.get("Sheets", [{}])[0].get("Items", [])

        for item in items:

            field = cls.clean(item.get("field"))
            value = item.get("value")

            if field == "applicant":
                applicant = value

            elif field in {"manufacturer", "manufacturer 1"}:
                manufacturer = value

        return applicant, manufacturer

    @classmethod
    def update(
        cls,
        cdr_json_path: str,
        trf_json_path: str,
        input_docx_path: str,
        output_docx_path: str
    ) -> None:
        """
        Update TRF JSON + DOCX using CDR values.
        """

        # Load files
        cdr_data = cls.load_json(cdr_json_path)
        trf_data = cls.load_json(trf_json_path)

        # Extract CDR values
        applicant, manufacturer = cls.extract_cdr_values(cdr_data)

        # Update TRF JSON
        for table in trf_data.get("Tables", []):

            for item in table.get("Items", []):

                field = cls.clean(item.get("field"))

                if field in cls.APPLICANT_FIELDS:
                    item["value"] = applicant

                elif field in cls.MANUFACTURER_FIELDS:
                    item["value"] = manufacturer

        # Save updated JSON
        cls.save_json(trf_data, trf_json_path)

        # Update DOCX
        doc = Document(input_docx_path)

        for table_obj in trf_data.get("Tables", []):

            table_index = table_obj.get("Table", -1)

            if table_index >= len(doc.tables):
                continue

            table = doc.tables[table_index]

            for item in table_obj.get("Items", []):

                editable = (
                    item.get("ai_fillable")
                    or item.get("task_type") == "verdict_dependency"
                    or item.get("user_editable")
                )

                value = item.get("value")

                if not editable or value is None:
                    continue

                try:
                    cell = table.cell(
                        item["answer_row"],
                        item["answer_column"]
                    )

                    cell.text = ""

                    run = cell.paragraphs[0].add_run(str(value))
                    run.font.name = "Arial"
                    run.font.size = Pt(10)

                    run._element.rPr.rFonts.set(
                        qn("w:eastAsia"),
                        "Arial"
                    )

                except Exception as e:
                    print(f"Error updating cell: {e}")

        doc.save(output_docx_path)

        print("TRF JSON and DOCX updated successfully.")

In [56]:
TRFUpdater.update(
    cdr_json_path=r"C:\Users\saurav\Downloads\cdr_payload_v5_updated 2.json",
    trf_json_path=r"D:\code\intertek\InterTek-AI-Repo_dev_Akshay_latest 4\Backend\data\input_files\pta_final_6_3_1.json",
    input_docx_path=r"D:\code\intertek\InterTek-AI-Repo_dev_Akshay_latest 4\Backend\data\input_files\input.docx",
    output_docx_path=r"D:\code\intertek\InterTek-AI-Repo_dev_Akshay_latest 4\Backend\data\input_files\output.docx",
)

TRF JSON and DOCX updated successfully.


In [1]:
import json

trf_json_path = r"D:\code\InterTek-AI-Repo\Backend\data\input_files\pta_final_6_3_1.json"

with open(trf_json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

In [3]:
letter_body = r"D:\code\InterTek-AI-Repo\Backend\data\G105000008\letter_body_iec_output_G105000008.json"
with open(trf_json_path, "r", encoding="utf-8") as f:
    letter_body_data = json.load(f)

letter_body_data

{'Tables': [{'Table': 0,
   'Items': [{'question_row': 0,
     'question_column': 3,
     'field': 'Test Report issued under the responsibility of:',
     'answer_row': 0,
     'answer_column': 4,
     'value': None,
     'task_type': None,
     'user_editable': True,
     'ai_fillable': False,
     'accuracy_level': False,
     'page_no': 1,
     'disable_text': True},
    {'question_row': 1,
     'question_column': 0,
     'field': 'TEST REPORT\nIEC 61010-1\nSafety requirements for electrical equipment for measurement, \ncontrol, and laboratory use\nPart 1: General requirements',
     'answer_row': 1,
     'answer_column': 2,
     'task_type': None,
     'user_editable': False,
     'ai_fillable': False,
     'accuracy_level': False,
     'page_no': 1,
     'disable_text': True},
    {'question_row': 3,
     'question_column': 0,
     'field': 'Report Number.\t:',
     'answer_row': 3,
     'answer_column': 2,
     'value': None,
     'task_type': None,
     'user_editable': True,
  

In [10]:
letter_body_data['Tables'][0]['Items'][0]['value']